In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSV files
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Helper to produce overview for a dataframe with timestamp column
def overview_df(df, ts_col='timestamp', id_col='cmdb_id', name_col=None, top_n=50):
    # Ensure timestamp is integer-like
    df[ts_col] = df[ts_col].astype('int64')
    min_ts = int(df[ts_col].min()) if not df.empty else None
    max_ts = int(df[ts_col].max()) if not df.empty else None
    min_dt_local = (pd.to_datetime(min_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
                    if min_ts is not None else None)
    max_dt_local = (pd.to_datetime(max_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
                    if max_ts is not None else None)
    unique_ids = df[id_col].value_counts().rename_axis(id_col).reset_index(name='count').head(top_n)
    name_counts = None
    if name_col is not None:
        name_counts = df[name_col].value_counts().rename_axis(name_col).reset_index(name='count').head(top_n)
    overview = pd.DataFrame({
        'metric': ['min_timestamp','max_timestamp','min_datetime_local','max_datetime_local',
                   'unique_cmdb_count','unique_name_count' if name_col is not None else 'unique_name_count'],
        'value': [min_ts, max_ts, min_dt_local, max_dt_local,
                  int(df[id_col].nunique()), int(df[name_col].nunique()) if name_col is not None else None]
    })
    return overview, unique_ids, name_counts

# Metric summary
metric_overview, metric_cmdbs, metric_kpi_counts = overview_df(metric_df, ts_col='timestamp',
                                                               id_col='cmdb_id', name_col='kpi_name', top_n=50)

# Trace summary
trace_overview, trace_cmdbs, trace_trace_counts = overview_df(trace_df, ts_col='timestamp',
                                                               id_col='cmdb_id', name_col='trace_name', top_n=50)

# Log summary
log_overview, log_cmdbs, log_log_counts = overview_df(log_df, ts_col='timestamp',
                                                      id_col='cmdb_id', name_col='log_name', top_n=50)

# Error logs summary: counts per cmdb_id and up to 5 sample rows
error_df['timestamp'] = error_df['timestamp'].astype('int64')
error_min_ts = int(error_df['timestamp'].min()) if not error_df.empty else None
error_max_ts = int(error_df['timestamp'].max()) if not error_df.empty else None
error_min_dt_local = (pd.to_datetime(error_min_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
                      if error_min_ts is not None else None)
error_max_dt_local = (pd.to_datetime(error_max_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
                      if error_max_ts is not None else None)
error_overview = pd.DataFrame({
    'metric': ['min_timestamp','max_timestamp','min_datetime_local','max_datetime_local','unique_cmdb_count','sample_rows_returned'],
    'value': [error_min_ts, error_max_ts, error_min_dt_local, error_max_dt_local, int(error_df['cmdb_id'].nunique()), min(5, len(error_df))]
})
error_cmdb_counts = error_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count')
error_samples = error_df[['timestamp','cmdb_id','message']].head(5)

# Display compact summaries (variables shown as outputs)
metric_overview, metric_cmdbs, metric_kpi_counts, trace_overview, trace_cmdbs, trace_trace_counts, log_overview, log_cmdbs, log_log_counts, error_overview, error_cmdb_counts, error_samples

```
Out[1]:
```
```python
# Summarize the previously computed overviews into a concise plain-English summary string.
# This cell reuses variables produced in the earlier run (metric_overview, metric_cmdbs, metric_kpi_counts, 
# trace_overview, trace_cmdbs, trace_trace_counts, log_overview, log_cmdbs, log_log_counts, error_overview).

def safe_get_overview_val(df, key):
    try:
        v = df.loc[df['metric']==key, 'value'].values[0]
        return v
    except Exception:
        return None

def top_list(df, col, n=10):
    if df is None or df.empty:
        return []
    return df[col].astype(str).head(n).tolist()

def top_name_counts(df, name_col, n=10):
    if df is None or df.empty:
        return []
    # combine name and count into "name(count)"
    rows = df.head(n)
    return [f"{r[name_col]}({int(r['count'])})" for _, r in rows.iterrows()]

# Metric summary values
m_min_ts = safe_get_overview_val(metric_overview, 'min_timestamp')
m_max_ts = safe_get_overview_val(metric_overview, 'max_timestamp')
m_min_dt = safe_get_overview_val(metric_overview, 'min_datetime_local')
m_max_dt = safe_get_overview_val(metric_overview, 'max_datetime_local')
m_unique_cmdb = safe_get_overview_val(metric_overview, 'unique_cmdb_count')
m_unique_names = safe_get_overview_val(metric_overview, 'unique_name_count')
m_top_cmdbs = top_list(metric_cmdbs, 'cmdb_id', n=10)
m_top_kpis = top_name_counts(metric_kpi_counts, 'kpi_name', n=10)

# Trace summary values
t_min_ts = safe_get_overview_val(trace_overview, 'min_timestamp')
t_max_ts = safe_get_overview_val(trace_overview, 'max_timestamp')
t_min_dt = safe_get_overview_val(trace_overview, 'min_datetime_local')
t_max_dt = safe_get_overview_val(trace_overview, 'max_datetime_local')
t_unique_cmdb = safe_get_overview_val(trace_overview, 'unique_cmdb_count')
t_unique_names = safe_get_overview_val(trace_overview, 'unique_name_count')
t_top_cmdbs = top_list(trace_cmdbs, 'cmdb_id', n=10)
t_top_traces = top_name_counts(trace_trace_counts, 'trace_name', n=10)

# Log summary values
l_min_ts = safe_get_overview_val(log_overview, 'min_timestamp')
l_max_ts = safe_get_overview_val(log_overview, 'max_timestamp')
l_min_dt = safe_get_overview_val(log_overview, 'min_datetime_local')
l_max_dt = safe_get_overview_val(log_overview, 'max_datetime_local')
l_unique_cmdb = safe_get_overview_val(log_overview, 'unique_cmdb_count')
l_unique_names = safe_get_overview_val(log_overview, 'unique_name_count')
l_top_cmdbs = top_list(log_cmdbs, 'cmdb_id', n=10)
l_top_logs = top_name_counts(log_log_counts, 'log_name', n=10)

# Error logs summary (error_overview exists)
e_min_ts = safe_get_overview_val(error_overview, 'min_timestamp')
e_max_ts = safe_get_overview_val(error_overview, 'max_timestamp')
e_min_dt = safe_get_overview_val(error_overview, 'min_datetime_local')
e_max_dt = safe_get_overview_val(error_overview, 'max_datetime_local')
e_unique_cmdb = safe_get_overview_val(error_overview, 'unique_cmdb_count')
# whether error samples exist:
has_error_samples = 'error_samples' in globals() and not error_samples.empty

# Build the final concise summary string
summary = (
    "Summary of telemetry files:\n\n"
    "1) metric.csv:\n"
    f"   - min_timestamp: {int(m_min_ts) if m_min_ts is not None and str(m_min_ts) != 'nan' else 'N/A'}"
    f", max_timestamp: {int(m_max_ts) if m_max_ts is not None and str(m_max_ts) != 'nan' else 'N/A'}\n"
    f"   - local time range (UTC+8): {m_min_dt} — {m_max_dt}\n"
    f"   - unique cmdb_id count: {int(m_unique_cmdb) if m_unique_cmdb is not None else 'N/A'}; sample top cmdb_ids: {', '.join(m_top_cmdbs) if m_top_cmdbs else 'N/A'}\n"
    f"   - unique kpi_name count: {int(m_unique_names) if m_unique_names is not None else 'N/A'}; top kpi_name counts: {', '.join(m_top_kpis) if m_top_kpis else 'N/A'}\n\n"
    "2) trace.csv:\n"
    f"   - min_timestamp: {int(t_min_ts) if t_min_ts is not None and str(t_min_ts) != 'nan' else 'N/A'}"
    f", max_timestamp: {int(t_max_ts) if t_max_ts is not None and str(t_max_ts) != 'nan' else 'N/A'}\n"
    f"   - local time range (UTC+8): {t_min_dt} — {t_max_dt}\n"
    f"   - unique cmdb_id count: {int(t_unique_cmdb) if t_unique_cmdb is not None else 'N/A'}; sample top cmdb_ids: {', '.join(t_top_cmdbs) if t_top_cmdbs else 'N/A'}\n"
    f"   - unique trace_name count: {int(t_unique_names) if t_unique_names is not None else 'N/A'}; top trace_name counts: {', '.join(t_top_traces) if t_top_traces else 'N/A'}\n\n"
    "3) log.csv:\n"
    f"   - min_timestamp: {int(l_min_ts) if l_min_ts is not None and str(l_min_ts) != 'nan' else 'N/A'}"
    f", max_timestamp: {int(l_max_ts) if l_max_ts is not None and str(l_max_ts) != 'nan' else 'N/A'}\n"
    f"   - local time range (UTC+8): {l_min_dt} — {l_max_dt}\n"
    f"   - unique cmdb_id count: {int(l_unique_cmdb) if l_unique_cmdb is not None else 'N/A'}; sample top cmdb_ids: {', '.join(l_top_cmdbs) if l_top_cmdbs else 'N/A'}\n"
    f"   - unique log_name count: {int(l_unique_names) if l_unique_names is not None else 'N/A'}; log_name counts: {', '.join(l_top_logs) if l_top_logs else 'N/A'}\n\n"
    "4) error_logs.csv:\n"
    f"   - min_timestamp: {e_min_ts if e_min_ts is not None and str(e_min_ts) != 'nan' else 'No rows'}"
    f", max_timestamp: {e_max_ts if e_max_ts is not None and str(e_max_ts) != 'nan' else 'No rows'}\n"
    f"   - unique cmdb_id count: {int(e_unique_cmdb) if e_unique_cmdb not in (None, 'nan') else 0}\n"
    f"   - samples present: {'Yes (up to 5 shown earlier)' if has_error_samples else 'No; file is empty'}\n\n"
    "Notes: times shown above were computed in UTC+8. The telemetry shows metric data spanning 08:30–09:30 (UTC+8),\n"
    "trace data and log data span ~09:00–09:30 (UTC+8). error_logs.csv contains no rows in this dataset."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               metric                     value
0       min_timestamp                1647822600
1       max_timestamp                1647826200
2  min_datetime_local  2022-03-21 08:30:00+0800
3  max_datetime_local  2022-03-21 09:30:00+0800
4   unique_cmdb_count                        59
5   unique_name_count                      1400,           cmdb_id  count
0       adservice  20801
1      adservice2  20313
2     frontend2-0  12200
3      frontend-2  11468
4      frontend-1  11468
..            ...    ...
45         node-5   3440
46         node-3   3440
47         node-4   3348
48  redis-cart2-0    732
49   redis-cart-0    732

[50 rows x 2 columns],                                              kpi_name  count
0   container.node-6.container_network_transmit_er...   2196
1   container.node-6.container_network_transmit_pa...   2196
2   container.node-6.container_network_transmit_pa...   2196
3   container.node-6.container_network_receive_MB....   2196
4   container.node-6.container_network_receive_err...   2196
..                                                ...    ...
45  container.node-6.container_fs_sector_reads./de...   2074
46    container.node-6.container_tasks_state.sleeping   2074
47   container.node-6.container_memory_working_set_MB   2074
48  container.node-6.container_fs_reads_merged./de...   2074
49    container.node-6.container_fs_reads_MB./dev/vda   2074

[50 rows x 2 columns],                metric                     value
0       min_timestamp                1647824400
1       max_timestamp                1647826140
2  min_datetime_local  2022-03-21 09:00:00+0800
3  max_datetime_local  2022-03-21 09:29:00+0800
4   unique_cmdb_count                        36
5   unique_name_count                       180,                      cmdb_id  count
0                 frontend-0   2632
1                 frontend-1   2620
2                 frontend-2   2464
3          checkoutservice-1   2036
4          checkoutservice-2   1964
..                       ...    ...
31  recommendationservice2-0     20
32            cartservice2-0      8
33              adservice2-0      4
34        currencyservice2-0      4
35  productcatalogservice2-0      4

[36 rows x 2 columns],                                   trace_name  count
0        trace.from_frontend-0.duration_mean    598
1         trace.from_frontend-0.duration_p95    598
2           trace.from_frontend-0.error_rate    598
3            trace.from_frontend-0.row_count    598
4        trace.from_frontend-1.duration_mean    595
..                                       ...    ...
45     trace.to_currencyservice-2.error_rate    173
46   trace.to_currencyservice-2.duration_p95    173
47  trace.to_currencyservice-2.duration_mean    173
48  trace.to_currencyservice-1.duration_mean    172
49   trace.to_currencyservice-1.duration_p95    172

[50 rows x 2 columns],                metric                     value
0       min_timestamp                1647824400
1       max_timestamp                1647826200
2  min_datetime_local  2022-03-21 09:00:00+0800
3  max_datetime_local  2022-03-21 09:30:00+0800
4   unique_cmdb_count                        31
5   unique_name_count                         2,               cmdb_id  count
0         adservice-1     62
1       cartservice-1     62
2       cartservice-0     62
3       cartservice-2     62
4   currencyservice-2     62
..                ...    ...
26  checkoutservice-2     56
27   paymentservice-1     56
28   paymentservice-2     56
29   paymentservice-0     56
30       redis-cart-0     10

[31 rows x 2 columns],           log_name  count
0  log.error_count    911
1    log.row_count    911,                  metric  value
0         min_timestamp    NaN
1         max_timestamp    NaN
2    min_datetime_local    NaN
3    max_datetime_local    NaN
4     unique_cmdb_count    0.0
5  sample_rows_returned    0.0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse metric_df loaded previously
# Ensure value is numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Compute global thresholds (P95, P5) and sample count for each (cmdb_id, kpi_name)
metric_thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(
        count=lambda x: x.dropna().shape[0],
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05)
    )
)

# Sort by highest p95 and return top 20 rows with the requested columns
metric_thresholds_top20 = metric_thresholds.sort_values('p95', ascending=False)[['cmdb_id', 'kpi_name', 'p95', 'p5', 'count']].head(20)

# Display the compact result (and keep metric_thresholds in the kernel for later steps)
metric_thresholds_top20

```
Out[2]:
```
```python
# Concise plain-English summary stored as a string and displayed.
summary = (
    "Global thresholds (P95, P5, count) were computed for every (cmdb_id, kpi_name) series.\n"
    "Top 20 series by highest P95 (descending) were returned in 'metric_thresholds_top20'.\n\n"
    "Key observations from the top 20:\n"
    " - Highest P95 values belong to adservice-related JVM/runtime metrics, e.g.:\n"
    "     * adservice | OperatingSystem_ProcessCpuTime: p95=2.174314e+12, p5=2.165068e+12, count=61\n"
    "     * adservice2/adservice | Runtime_StartTime: p95 ~1.6478e+12, count=61\n"
    "     * adservice | various Threading/CPU time metrics with p95 in the 1e11–2e11 range, count=61\n"
    " - Several node-level filesystem/disk metrics (system.disk.total, system.fs.inodes.total/free, system.disk.used)\n"
    "   appear in the top 20 with p95 values around 1.1e10–1.8e10. Most of these series have count=61,\n"
    "   but a few node-4 series show only count=44 (fewer samples).\n\n"
    "Storage: the full aggregated table is stored in 'metric_thresholds' for later steps; 'metric_thresholds_top20' holds the shown top-20 subset."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name           p95            p5  count
244    adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime  2.174314e+12  2.165068e+12     61
857   adservice2                runtime.java_lang_Runtime_StartTime  1.647819e+12  1.647819e+12     61
250    adservice                runtime.java_lang_Runtime_StartTime  1.647144e+12  1.647144e+12     61
253    adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  2.284479e+11  2.273622e+11     61
255    adservice  runtime.java_lang_Threading_CurrentThreadUserTime  2.085325e+11  2.074833e+11     61
252    adservice  runtime.java_lang_Threading_CurrentThreadAlloc...  1.652630e+11  1.644784e+11     61
851   adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime  3.249333e+10  2.185000e+10     61
3684      node-5                                  system.disk.total  1.854945e+10  1.854945e+10     61
3688      node-5                             system.fs.inodes.total  1.525913e+10  1.516983e+10     61
3686      node-5                              system.fs.inodes.free  1.525450e+10  1.516520e+10     61
3685      node-5                                   system.disk.used  1.464770e+10  1.462530e+10     61
3450      node-1                                  system.disk.total  1.406688e+10  1.406688e+10     61
3567      node-3                                  system.disk.total  1.406688e+10  1.406688e+10     61
3626      node-4                                  system.disk.total  1.406688e+10  1.406688e+10     44
3509      node-2                                  system.disk.total  1.406688e+10  1.406688e+10     61
3630      node-4                             system.fs.inodes.total  1.145422e+10  1.138785e+10     44
3454      node-1                             system.fs.inodes.total  1.145422e+10  1.138744e+10     61
3513      node-2                             system.fs.inodes.total  1.145404e+10  1.138728e+10     61
3571      node-3                             system.fs.inodes.total  1.145402e+10  1.138733e+10     61
3452      node-1                              system.fs.inodes.free  1.145072e+10  1.138394e+10     61```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# Reuse existing variables: metric_df, metric_thresholds
# Ensure necessary columns have correct dtypes
metric_df['timestamp'] = metric_df['timestamp'].astype('int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Prepare timezone and convert local window to UTC epoch seconds
tz = pytz.timezone('Asia/Shanghai')
start_local = pd.Timestamp('2022-03-21 09:00:00').tz_localize(tz)
end_local = pd.Timestamp('2022-03-21 09:30:00').tz_localize(tz)
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts = int(end_local.astimezone(pytz.utc).timestamp())

# Filter metric data to the incident window (inclusive)
metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# Prepare thresholds table: rename global count to avoid confusion
thresholds = metric_thresholds.rename(columns={'count': 'global_count'})

# Join filtered rows with global thresholds on (cmdb_id, kpi_name)
merged = metric_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='inner')

# If merged is empty, create an empty metric_window_anomalies with expected columns
if merged.empty:
    metric_window_anomalies = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points_in_window','anomaly_count_high','anomaly_count_low',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','min_value_in_window','p95','p5','global_count'
    ])
else:
    # Compute anomaly flags
    merged['is_high'] = merged['value'] >= merged['p95']
    merged['is_low'] = merged['value'] <= merged['p5']
    merged['is_anomaly'] = merged['is_high'] | merged['is_low']

    # Aggregate per (cmdb_id, kpi_name, p95, p5)
    def agg_group(g):
        total = int(g.shape[0])
        high = int(g['is_high'].sum())
        low = int(g['is_low'].sum())
        if g['is_anomaly'].any():
            earliest = int(g.loc[g['is_anomaly'], 'timestamp'].min())
            latest = int(g.loc[g['is_anomaly'], 'timestamp'].max())
        else:
            earliest = pd.NA
            latest = pd.NA
        return pd.Series({
            'total_points_in_window': total,
            'anomaly_count_high': high,
            'anomaly_count_low': low,
            'earliest_anomaly_timestamp': earliest,
            'latest_anomaly_timestamp': latest,
            'max_value_in_window': g['value'].max(),
            'min_value_in_window': g['value'].min()
        })

    grouped = merged.groupby(['cmdb_id', 'kpi_name', 'p95', 'p5', 'global_count'], as_index=False).apply(lambda df: agg_group(df))
    # The groupby+apply produced a multiindex; reset index appropriately
    grouped = grouped.reset_index()
    # Keep/rename columns to a flat structure
    metric_window_anomalies = grouped[[
        'cmdb_id','kpi_name','total_points_in_window','anomaly_count_high','anomaly_count_low',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','min_value_in_window','p95','p5','global_count'
    ]].copy()

# Sort by total anomalies desc then by max_value_in_window desc
metric_window_anomalies['total_anomalies'] = metric_window_anomalies['anomaly_count_high'].fillna(0).astype(int) + metric_window_anomalies['anomaly_count_low'].fillna(0).astype(int)
metric_window_anomalies_sorted = metric_window_anomalies.sort_values(['total_anomalies', 'max_value_in_window'], ascending=[False, False])

# Select requested columns and limit to top 20 for compact output
metric_window_anomalies_top20 = metric_window_anomalies_sorted[[
    'cmdb_id','kpi_name','total_points_in_window','anomaly_count_high','anomaly_count_low',
    'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','min_value_in_window','p95','p5'
]].head(20)

# Keep full result in kernel variable 'metric_window_anomalies' (already set) and display the compact top20
metric_window_anomalies, metric_window_anomalies_top20

```
Out[3]:
```
```python
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use existing aggregated variables from the kernel
# metric_window_anomalies and metric_window_anomalies_top20 should exist
total_series = int(metric_window_anomalies.shape[0]) if 'metric_window_anomalies' in globals() else 0

top20 = metric_window_anomalies_top20.copy() if 'metric_window_anomalies_top20' in globals() else pd.DataFrame()

def fmt_ts(ts):
    if pd.isna(ts):
        return "N/A"
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

summary_lines = []
summary_lines.append(f"Window: 2022-03-21 09:00:00 → 2022-03-21 09:30:00 (UTC+8)")
summary_lines.append(f"Aggregated series (cmdb_id, kpi_name) in window: {total_series}")
summary_lines.append("Top 20 series by anomaly count are stored in 'metric_window_anomalies_top20'.")
summary_lines.append("Top examples (from the top-20 list):")

# compute total anomalies for the top20 rows and pick top 3 examples
if not top20.empty:
    top20 = top20.copy()
    top20['anomaly_count_high'] = top20['anomaly_count_high'].fillna(0).astype(int)
    top20['anomaly_count_low'] = top20['anomaly_count_low'].fillna(0).astype(int)
    top20['total_anomalies'] = top20['anomaly_count_high'] + top20['anomaly_count_low']
    examples = top20.sort_values(['total_anomalies','max_value_in_window'], ascending=[False, False]).head(3)
    for i, row in examples.iterrows():
        cmdb = row['cmdb_id']
        kpi = row['kpi_name']
        total_pts = int(row['total_points_in_window']) if not pd.isna(row['total_points_in_window']) else 0
        high = int(row['anomaly_count_high'])
        low = int(row['anomaly_count_low'])
        earliest = fmt_ts(row['earliest_anomaly_timestamp'])
        latest = fmt_ts(row['latest_anomaly_timestamp'])
        maxv = float(row['max_value_in_window']) if not pd.isna(row['max_value_in_window']) else None
        minv = float(row['min_value_in_window']) if not pd.isna(row['min_value_in_window']) else None
        p95 = float(row['p95']) if not pd.isna(row['p95']) else None
        p5 = float(row['p5']) if not pd.isna(row['p5']) else None
        summary_lines.append(
            f"- {cmdb} | {kpi}: points={total_pts}, high_anomalies={high}, low_anomalies={low}, "
            f"earliest={earliest}, latest={latest}, max={maxv}, min={minv}, p95={p95}, p5={p5}"
        )
else:
    summary_lines.append("- No series found in the window or top20 is empty.")

summary_lines.append("")
summary_lines.append("Note: Full aggregated table is stored in 'metric_window_anomalies' for later steps.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name  total_points_in_window  anomaly_count_high  anomaly_count_low earliest_anomaly_timestamp latest_anomaly_timestamp  max_value_in_window  min_value_in_window         p95          p5  global_count  total_anomalies
0              adservice                                     app.grpc.count                    31.0                 3.0                2.0               1647825060.0             1647826140.0           109.000000            43.000000   93.000000   60.000000            61                5
1              adservice                                       app.grpc.mrt                    31.0                 3.0                0.0               1647824940.0             1647826200.0             0.008961             0.002364    0.002927    0.002333            61                3
2              adservice                                        app.grpc.rr                    31.0                31.0               31.0               1647824400.0             1647826200.0           100.000000           100.000000  100.000000  100.000000            61               62
3              adservice                                        app.grpc.sr                    31.0                30.0               31.0               1647824400.0             1647826200.0           100.000000            99.082569  100.000000  100.000000            61               61
4              adservice                                     app.http.count                    31.0                31.0               31.0               1647824400.0             1647826200.0            24.000000            24.000000   24.000000   24.000000            61               62
...                  ...                                                ...                     ...                 ...                ...                        ...                      ...                  ...                  ...         ...         ...           ...              ...
5293  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...                    31.0                31.0               30.0               1647824400.0             1647826200.0             1.000000             0.000000    0.000000    0.000000            61               61
5294  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....                    31.0                31.0               30.0               1647824400.0             1647826200.0          1325.000000             0.000000    0.000000    0.000000            61               61
5295  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....                    31.0                31.0               30.0               1647824400.0             1647826200.0             0.000525             0.000000    0.000000    0.000000            61               61
5296  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....                    31.0                31.0               30.0               1647824400.0             1647826200.0             0.500000             0.000000    0.000000    0.000000            61               61
5297  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....                    31.0                31.0               30.0               1647824400.0             1647826200.0            49.750000             0.000000    0.000000    0.000000            61               61

[5298 rows x 13 columns],                       cmdb_id                                           kpi_name  total_points_in_window  anomaly_count_high  anomaly_count_low earliest_anomaly_timestamp latest_anomaly_timestamp  max_value_in_window  min_value_in_window           p95            p5
857                adservice2                runtime.java_lang_Runtime_StartTime                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.647819e+12         1.647819e+12  1.647819e+12  1.647819e+12
250                 adservice                runtime.java_lang_Runtime_StartTime                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.647144e+12         1.647144e+12  1.647144e+12  1.647144e+12
3684                   node-5                                  system.disk.total                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.854945e+10         1.854945e+10  1.854945e+10  1.854945e+10
3450                   node-1                                  system.disk.total                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.406688e+10         1.406688e+10  1.406688e+10  1.406688e+10
3509                   node-2                                  system.disk.total                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.406688e+10         1.406688e+10  1.406688e+10  1.406688e+10
3567                   node-3                                  system.disk.total                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.406688e+10         1.406688e+10  1.406688e+10  1.406688e+10
238                 adservice  runtime.java_lang_OperatingSystem_CommittedVir...                    31.0                31.0               31.0               1647824400.0             1647826200.0         3.591926e+09         3.591926e+09  3.591926e+09  3.591926e+09
1003             adservice2-0      container.node-6.container_start_time_seconds                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.647819e+09         1.647819e+09  1.647819e+09  1.647819e+09
1405           cartservice2-0      container.node-5.container_start_time_seconds                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.647782e+09         1.647782e+09  1.647782e+09  1.647782e+09
1109            cartservice-0      container.node-6.container_start_time_seconds                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.647760e+09         1.647760e+09  1.647760e+09  1.647760e+09
1305            cartservice-2      container.node-6.container_start_time_seconds                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.647759e+09         1.647759e+09  1.647759e+09  1.647759e+09
1209            cartservice-1      container.node-6.container_start_time_seconds                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.647757e+09         1.647757e+09  1.647757e+09  1.647757e+09
3993         paymentservice-2      container.node-6.container_start_time_seconds                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.647318e+09         1.647318e+09  1.647318e+09  1.647318e+09
2333       currencyservice2-0      container.node-6.container_start_time_seconds                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.647236e+09         1.647236e+09  1.647236e+09  1.647236e+09
2245        currencyservice-2      container.node-6.container_start_time_seconds                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.647236e+09         1.647236e+09  1.647236e+09  1.647236e+09
3109               frontend-2      container.node-5.container_start_time_seconds                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.647144e+09         1.647144e+09  1.647144e+09  1.647144e+09
4801  recommendationservice-2      container.node-6.container_start_time_seconds                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.647144e+09         1.647144e+09  1.647144e+09  1.647144e+09
1793        checkoutservice-2      container.node-5.container_start_time_seconds                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.647144e+09         1.647144e+09  1.647144e+09  1.647144e+09
2921               frontend-1      container.node-5.container_start_time_seconds                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.647144e+09         1.647144e+09  1.647144e+09  1.647144e+09
5189        shippingservice-2      container.node-5.container_start_time_seconds                    31.0                31.0               31.0               1647824400.0             1647826200.0         1.647144e+09         1.647144e+09  1.647144e+09  1.647144e+09)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_window_anomalies from kernel
df = metric_window_anomalies.copy() if 'metric_window_anomalies' in globals() else pd.DataFrame()

# Ensure numeric types for relevant columns
num_cols = ['anomaly_count_high','anomaly_count_low','earliest_anomaly_timestamp',
            'latest_anomaly_timestamp','max_value_in_window','p95']
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

# If empty, create empty summary with expected columns
if df.empty:
    component_anomaly_summary = pd.DataFrame(columns=[
        'cmdb_id','total_anomalies','num_kpis_with_anomaly',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_excess_pct'
    ])
else:
    # compute per-row total anomalies
    df['row_total_anomalies'] = df[['anomaly_count_high','anomaly_count_low']].fillna(0).sum(axis=1).astype(int)

    # compute per-row excess_pct where p95 != 0 and not null
    def compute_excess(row):
        p95 = row['p95']
        maxv = row['max_value_in_window']
        if pd.isna(p95) or p95 == 0 or pd.isna(maxv):
            return np.nan
        return (maxv - p95) / p95

    df['excess_pct'] = df.apply(compute_excess, axis=1)

    # Group by cmdb_id and aggregate required metrics
    agg = df.groupby('cmdb_id').apply(
        lambda g: pd.Series({
            'total_anomalies': int(g['row_total_anomalies'].sum()),
            'num_kpis_with_anomaly': int(g.loc[g['row_total_anomalies']>0, 'kpi_name'].nunique()),
            'earliest_anomaly_timestamp': int(g['earliest_anomaly_timestamp'].dropna().min()) if g['earliest_anomaly_timestamp'].dropna().size>0 else pd.NA,
            'latest_anomaly_timestamp': int(g['latest_anomaly_timestamp'].dropna().max()) if g['latest_anomaly_timestamp'].dropna().size>0 else pd.NA,
            'max_excess_pct': float(g['excess_pct'].dropna().max()) if g['excess_pct'].dropna().size>0 else np.nan
        })
    ).reset_index()

    component_anomaly_summary = agg.copy()

# Sort and prepare compact top-20 view
component_anomaly_summary_sorted = component_anomaly_summary.sort_values(
    by=['total_anomalies','max_excess_pct'], ascending=[False, False]
)

component_anomaly_summary_top20 = component_anomaly_summary_sorted[[
    'cmdb_id','total_anomalies','num_kpis_with_anomaly',
    'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_excess_pct'
]].head(20)

# Keep full result in kernel variable and display compact top20
component_anomaly_summary, component_anomaly_summary_top20

```
Out[4]:
```
```python
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variable
df = component_anomaly_summary.copy() if 'component_anomaly_summary' in globals() else pd.DataFrame()
top20 = component_anomaly_summary_top20.copy() if 'component_anomaly_summary_top20' in globals() else pd.DataFrame()

def fmt_ts(ts):
    try:
        if pd.isna(ts):
            return "N/A"
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    except Exception:
        return "N/A"

# Basic counts
num_components = int(df.shape[0]) if not df.empty else 0

# Prepare top-5 summary lines
top_lines = []
if not top20.empty:
    top_n = top20.head(5)
    for _, r in top_n.iterrows():
        cmdb = r['cmdb_id']
        total = int(r['total_anomalies']) if not pd.isna(r['total_anomalies']) else 0
        num_kpis = int(r['num_kpis_with_anomaly']) if not pd.isna(r['num_kpis_with_anomaly']) else 0
        earliest = fmt_ts(r['earliest_anomaly_timestamp'])
        latest = fmt_ts(r['latest_anomaly_timestamp'])
        max_excess = r['max_excess_pct']
        max_excess_str = f"{max_excess:.2f}" if pd.notna(max_excess) else "N/A"
        top_lines.append(f"{cmdb}: total_anomalies={total}, kpis_with_anomaly={num_kpis}, earliest={earliest}, latest={latest}, max_excess_pct={max_excess_str}")

summary = (
    f"Aggregated anomalies by cmdb_id:\n"
    f"- Components analyzed: {num_components}\n"
    f"- Top components (by total_anomalies):\n"
    + ("\n".join(f"  - {line}" for line in top_lines) if top_lines else "  - No data available")
    + "\n\nNote: Full per-component table is stored in 'component_anomaly_summary'.\n"
    "Displayed top entries come from 'component_anomaly_summary_top20'. Times are in UTC+8."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id  total_anomalies  num_kpis_with_anomaly  earliest_anomaly_timestamp  latest_anomaly_timestamp  max_excess_pct
0            adservice          18906.0                  341.0                1.647824e+09              1.647826e+09       10.000000
1          adservice-0           4432.0                   94.0                1.647824e+09              1.647826e+09        0.890259
2          adservice-1           3900.0                   86.0                1.647824e+09              1.647826e+09        6.655019
3          adservice-2           4691.0                   94.0                1.647824e+09              1.647826e+09        0.400000
4           adservice2          17707.0                  333.0                1.647824e+09              1.647826e+09        6.220873
..                 ...              ...                    ...                         ...                       ...             ...
54     shippingservice            135.0                    4.0                1.647824e+09              1.647826e+09        0.099058
55   shippingservice-0           3356.0                   80.0                1.647824e+09              1.647826e+09       14.364379
56   shippingservice-1           3368.0                   80.0                1.647824e+09              1.647826e+09       15.657930
57   shippingservice-2           3553.0                   80.0                1.647824e+09              1.647826e+09        7.809927
58  shippingservice2-0           4600.0                   84.0                1.647824e+09              1.647826e+09        9.281093

[59 rows x 6 columns],                      cmdb_id  total_anomalies  num_kpis_with_anomaly  earliest_anomaly_timestamp  latest_anomaly_timestamp  max_excess_pct
0                  adservice          18906.0                  341.0                1.647824e+09              1.647826e+09       10.000000
4                 adservice2          17707.0                  333.0                1.647824e+09              1.647826e+09        6.220873
30               frontend2-0          11723.0                  200.0                1.647824e+09              1.647826e+09       16.580427
15        checkoutservice2-0           8352.0                  148.0                1.647824e+09              1.647826e+09      336.000000
28                frontend-1           8068.0                  188.0                1.647824e+09              1.647826e+09        4.939244
29                frontend-2           7964.0                  188.0                1.647824e+09              1.647826e+09        4.067466
27                frontend-0           7927.0                  188.0                1.647824e+09              1.647826e+09        2.064247
51  recommendationservice2-0           5701.0                  102.0                1.647824e+09              1.647826e+09        1.778972
13         checkoutservice-1           5522.0                  144.0                1.647824e+09              1.647826e+09       97.333333
46  productcatalogservice2-0           5396.0                  104.0                1.647824e+09              1.647826e+09       26.160000
10            cartservice2-0           5389.0                  100.0                1.647824e+09              1.647826e+09       20.880435
5               adservice2-0           5380.0                  102.0                1.647824e+09              1.647826e+09       10.666667
14         checkoutservice-2           5236.0                  140.0                1.647824e+09              1.647826e+09      289.333333
12         checkoutservice-0           5111.0                  140.0                1.647824e+09              1.647826e+09       10.104244
44   productcatalogservice-1           5049.0                  120.0                1.647824e+09              1.647826e+09        4.244444
43   productcatalogservice-0           5010.0                  120.0                1.647824e+09              1.647826e+09        3.305556
20        currencyservice2-0           4777.0                   84.0                1.647824e+09              1.647826e+09        6.898857
3                adservice-2           4691.0                   94.0                1.647824e+09              1.647826e+09        0.400000
45   productcatalogservice-2           4609.0                  116.0                1.647824e+09              1.647826e+09      473.666667
58        shippingservice2-0           4600.0                   84.0                1.647824e+09              1.647826e+09        9.281093)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing trace_df if available; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure correct dtypes
trace_df['timestamp'] = trace_df['timestamp'].astype('int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Timezone and incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = pd.Timestamp('2022-03-21 09:00:00').tz_localize(tz)
end_local = pd.Timestamp('2022-03-21 09:30:00').tz_localize(tz)
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts = int(end_local.astimezone(pytz.utc).timestamp())

# 1) Compute global thresholds (P95, P5, count) across full trace data BEFORE filtering
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(
        count=lambda x: x.dropna().shape[0],
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05)
    )
)

# 2) Filter trace data to the incident window (inclusive)
trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Join filtered rows with global thresholds on (cmdb_id, trace_name)
merged = trace_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='inner')

# 4) Compute anomaly flags and aggregate per (cmdb_id, trace_name)
if merged.empty:
    # create empty DataFrame with expected columns
    trace_window_anomalies = pd.DataFrame(columns=[
        'cmdb_id','trace_name','total_points_in_window','anomaly_count_high','anomaly_count_low',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','min_value_in_window','p95','p5'
    ])
else:
    merged['is_high'] = merged['value'] >= merged['p95']
    merged['is_low'] = merged['value'] <= merged['p5']
    merged['is_anomaly'] = merged['is_high'] | merged['is_low']
    # anomaly timestamp only where anomaly
    merged['anomaly_ts'] = merged['timestamp'].where(merged['is_anomaly'], pd.NA)

    grouped = (
        merged
        .groupby(['cmdb_id','trace_name','p95','p5'], as_index=False)
        .agg(
            total_points_in_window=('value','count'),
            anomaly_count_high=('is_high','sum'),
            anomaly_count_low=('is_low','sum'),
            earliest_anomaly_timestamp=('anomaly_ts','min'),
            latest_anomaly_timestamp=('anomaly_ts','max'),
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min')
        )
    )

    # Normalize types: ensure integer timestamps where present
    def to_int_or_na(x):
        if pd.isna(x):
            return pd.NA
        return int(x)

    grouped['earliest_anomaly_timestamp'] = grouped['earliest_anomaly_timestamp'].apply(to_int_or_na)
    grouped['latest_anomaly_timestamp'] = grouped['latest_anomaly_timestamp'].apply(to_int_or_na)
    # Ensure counts are ints
    grouped['total_points_in_window'] = grouped['total_points_in_window'].astype(int)
    grouped['anomaly_count_high'] = grouped['anomaly_count_high'].astype(int)
    grouped['anomaly_count_low'] = grouped['anomaly_count_low'].astype(int)

    # Store full aggregated result
    trace_window_anomalies = grouped[[
        'cmdb_id','trace_name','total_points_in_window','anomaly_count_high','anomaly_count_low',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','min_value_in_window','p95','p5'
    ]].copy()

# 5) Prepare compact top-20 table sorted by total anomalies desc then max_value_in_window desc
trace_window_anomalies['total_anomalies'] = trace_window_anomalies['anomaly_count_high'].fillna(0).astype(int) + trace_window_anomalies['anomaly_count_low'].fillna(0).astype(int)
trace_window_anomalies_top20 = trace_window_anomalies.sort_values(['total_anomalies','max_value_in_window'], ascending=[False, False]).head(20)
# Keep trace_thresholds and trace_window_anomalies in kernel for later steps

# Display compact top-20 result
trace_window_anomalies_top20

```
Out[5]:
```
```python
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use existing variables from kernel if present
tt = trace_thresholds if 'trace_thresholds' in globals() else None
twa = trace_window_anomalies if 'trace_window_anomalies' in globals() else None
twa_top20 = trace_window_anomalies_top20 if 'trace_window_anomalies_top20' in globals() else None

def fmt_ts(ts):
    if pd.isna(ts):
        return "N/A"
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

if tt is None:
    summary = "trace_thresholds was not found in the kernel."
elif twa is None:
    summary = "trace_window_anomalies was not found in the kernel."
else:
    total_threshold_series = int(tt.shape[0])
    total_window_series = int(twa.shape[0])
    summary_lines = [
        "Trace thresholds and window anomalies summary:",
        f"- Global thresholds computed and stored in 'trace_thresholds' for {total_threshold_series} (cmdb_id, trace_name) series.",
        f"- Window aggregated anomalies stored in 'trace_window_anomalies' for {total_window_series} series (window: 2022-03-21 09:00 → 09:30 UTC+8).",
        "- Top-20 anomalous trace series (sorted by anomaly count) show a strong concentration in error_rate traces."
    ]

    # Show up to 5 example rows from top20
    if twa_top20 is None or twa_top20.empty:
        summary_lines.append("- No top-20 trace anomalies to display.")
    else:
        summary_lines.append("- Examples (up to 5) from trace_window_anomalies_top20:")
        examples = twa_top20.head(5).copy()
        # ensure numeric fields
        for _, r in examples.iterrows():
            cmdb = r['cmdb_id']
            trace_name = r['trace_name']
            total_pts = int(r['total_points_in_window'])
            high = int(r['anomaly_count_high'])
            low = int(r['anomaly_count_low'])
            earliest = fmt_ts(r['earliest_anomaly_timestamp'])
            latest = fmt_ts(r['latest_anomaly_timestamp'])
            maxv = float(r['max_value_in_window'])
            minv = float(r['min_value_in_window'])
            p95 = float(r['p95'])
            p5 = float(r['p5'])
            summary_lines.append(
                f"  - {cmdb} | {trace_name}: points={total_pts}, high={high}, low={low}, "
                f"earliest={earliest}, latest={latest}, max={maxv}, min={minv}, p95={p95}, p5={p5}"
            )

        summary_lines.append(
            "- Observation: many top entries are 'trace.*.error_rate' with p95=0 and p5=0, resulting in all window values flagged as both >=p95 and <=p5 (counts reflect this)."
        )

    summary_lines.append("- Notes: thresholds were computed globally across the entire trace.csv before filtering to the incident window (per requirement).")
    summary_lines.append("Stored variables: 'trace_thresholds' and 'trace_window_anomalies' are available for follow-up analysis.")
    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                        trace_name  total_points_in_window  anomaly_count_high  anomaly_count_low  earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window  min_value_in_window  p95   p5  total_anomalies
2      adservice-0  trace.from_frontend-0.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
6      adservice-0  trace.from_frontend-1.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
10     adservice-0  trace.from_frontend-2.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
14     adservice-1  trace.from_frontend-0.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
18     adservice-1  trace.from_frontend-1.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
22     adservice-1  trace.from_frontend-2.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
26     adservice-2  trace.from_frontend-0.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
30     adservice-2  trace.from_frontend-1.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
34     adservice-2  trace.from_frontend-2.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
54   cartservice-0  trace.from_frontend-0.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
58   cartservice-0  trace.from_frontend-1.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
62   cartservice-0  trace.from_frontend-2.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
66   cartservice-0             trace.self.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
82   cartservice-1  trace.from_frontend-0.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
86   cartservice-1  trace.from_frontend-1.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
90   cartservice-1  trace.from_frontend-2.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
94   cartservice-1             trace.self.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
110  cartservice-2  trace.from_frontend-0.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
114  cartservice-2  trace.from_frontend-1.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
118  cartservice-2  trace.from_frontend-2.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse trace_window_anomalies from kernel
twa = trace_window_anomalies.copy() if 'trace_window_anomalies' in globals() else pd.DataFrame()

# Ensure earliest_anomaly_timestamp is present and numeric, exclude NaN
if 'earliest_anomaly_timestamp' in twa.columns:
    twa_valid = twa[twa['earliest_anomaly_timestamp'].notna()].copy()
else:
    twa_valid = pd.DataFrame(columns=twa.columns)

# If there are valid rows, group by earliest_anomaly_timestamp
tz = pytz.timezone('Asia/Shanghai')

if not twa_valid.empty:
    # Ensure integer timestamps
    twa_valid['earliest_anomaly_timestamp'] = twa_valid['earliest_anomaly_timestamp'].astype(int)

    def sample_cmdbs(series):
        # preserve order of appearance, take up to 10 unique cmdb_ids
        uniques = pd.unique(series)
        return ",".join([str(x) for x in uniques[:10]])

    grouped = (
        twa_valid
        .groupby('earliest_anomaly_timestamp', as_index=False)
        .agg(
            series_count=('trace_name', 'nunique'),
            sample_cmdb_ids=('cmdb_id', sample_cmdbs)
        )
    )

    # Add datetime_local column (UTC+8)
    grouped['datetime_local'] = grouped['earliest_anomaly_timestamp'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    )

    # Reorder columns
    trace_earliest_ts_groups = grouped[[
        'earliest_anomaly_timestamp','datetime_local','series_count','sample_cmdb_ids'
    ]].copy()

    # Sort by series_count desc, then earliest_anomaly_timestamp asc
    trace_earliest_ts_groups = trace_earliest_ts_groups.sort_values(['series_count','earliest_anomaly_timestamp'], ascending=[False, True]).reset_index(drop=True)
else:
    # empty result
    trace_earliest_ts_groups = pd.DataFrame(columns=[
        'earliest_anomaly_timestamp','datetime_local','series_count','sample_cmdb_ids'
    ])

# Prepare compact top-10 view
trace_earliest_ts_groups_top10 = trace_earliest_ts_groups.head(10)

# Store full result in kernel variable and display compact top10
trace_earliest_ts_groups, trace_earliest_ts_groups_top10

```
Out[6]:
```
```python
# Produce a concise plain-English summary of the grouped earliest anomaly timestamps.
# Reuse the previously stored 'trace_earliest_ts_groups' / top10 view from the kernel.

try:
    tbl = trace_earliest_ts_groups.head(10).copy()
except Exception:
    tbl = pd.DataFrame()

if tbl.empty:
    summary = "No earliest-anomaly timestamp groups available."
else:
    lines = []
    lines.append("Earliest-anomaly timestamp groups (top 10 by series_count):")
    for _, r in tbl.iterrows():
        ts = int(r['earliest_anomaly_timestamp'])
        dt_local = r.get('datetime_local', pd.to_datetime(ts, unit='s', utc=True).tz_convert(pytz.timezone('Asia/Shanghai')).strftime('%Y-%m-%d %H:%M:%S%z'))
        count = int(r['series_count'])
        sample_cmdbs = r['sample_cmdb_ids']
        # Keep sample list compact (already up to 10). Truncate if it's too long.
        if isinstance(sample_cmdbs, str) and len(sample_cmdbs) > 200:
            sample_cmdbs = sample_cmdbs[:197] + "..."
        lines.append(f"- {dt_local} (ts={ts}): series_count={count}; sample_cmdb_ids={sample_cmdbs}")
    lines.append("")
    lines.append("Top observations:")
    lines.append("- The largest cluster occurred at 2022-03-21 09:00:00+08 with 104 trace series having their earliest anomaly at that time.")
    lines.append("- Subsequent peaks occur at 09:01 (+72 series), 09:10 (+53), and 09:29 (+48).")
    lines.append("- Many involved cmdb_ids include adservice-0/1/2 and various cartservice/checkouts/frontends, indicating broad multi-service impact starting at 09:00 (UTC+8).")
    lines.append("")
    lines.append("Stored variable: full grouped table is in 'trace_earliest_ts_groups' for further analysis.")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    earliest_anomaly_timestamp            datetime_local  series_count                                    sample_cmdb_ids
0                   1647824400  2022-03-21 09:00:00+0800           104  adservice-0,adservice-1,adservice-2,cartservic...
1                   1647824460  2022-03-21 09:01:00+0800            72  adservice-0,adservice-1,adservice-2,cartservic...
2                   1647825000  2022-03-21 09:10:00+0800            53  adservice-0,adservice-1,adservice-2,cartservic...
3                   1647826140  2022-03-21 09:29:00+0800            48  adservice2-0,cartservice2-0,currencyservice2-0...
4                   1647824520  2022-03-21 09:02:00+0800            42  adservice-0,adservice-2,cartservice-0,cartserv...
5                   1647824580  2022-03-21 09:03:00+0800            41  adservice-0,adservice-1,cartservice-0,cartserv...
6                   1647824640  2022-03-21 09:04:00+0800            30  adservice-2,checkoutservice-0,checkoutservice-...
7                   1647824700  2022-03-21 09:05:00+0800            29  adservice-2,checkoutservice-0,checkoutservice-...
8                   1647824940  2022-03-21 09:09:00+0800            29  cartservice-1,cartservice-2,checkoutservice-0,...
9                   1647824820  2022-03-21 09:07:00+0800            27  adservice-0,adservice-1,cartservice-1,checkout...
10                  1647824880  2022-03-21 09:08:00+0800            20  checkoutservice-0,checkoutservice-1,checkoutse...
11                  1647825060  2022-03-21 09:11:00+0800            18  checkoutservice-1,checkoutservice-2,frontend-0...
12                  1647825300  2022-03-21 09:15:00+0800            11  adservice-2,checkoutservice-1,checkoutservice-...
13                  1647824760  2022-03-21 09:06:00+0800             7  checkoutservice-1,checkoutservice-2,currencyse...
14                  1647825540  2022-03-21 09:19:00+0800             6  checkoutservice-0,emailservice-0,productcatalo...
15                  1647825600  2022-03-21 09:20:00+0800             4                   checkoutservice-1,emailservice-2
16                  1647825900  2022-03-21 09:25:00+0800             4                   checkoutservice-1,emailservice-1
17                  1647825240  2022-03-21 09:14:00+0800             2                checkoutservice-1,shippingservice-1
18                  1647825420  2022-03-21 09:17:00+0800             2    productcatalogservice-1,recommendationservice-2,    earliest_anomaly_timestamp            datetime_local  series_count                                    sample_cmdb_ids
0                  1647824400  2022-03-21 09:00:00+0800           104  adservice-0,adservice-1,adservice-2,cartservic...
1                  1647824460  2022-03-21 09:01:00+0800            72  adservice-0,adservice-1,adservice-2,cartservic...
2                  1647825000  2022-03-21 09:10:00+0800            53  adservice-0,adservice-1,adservice-2,cartservic...
3                  1647826140  2022-03-21 09:29:00+0800            48  adservice2-0,cartservice2-0,currencyservice2-0...
4                  1647824520  2022-03-21 09:02:00+0800            42  adservice-0,adservice-2,cartservice-0,cartserv...
5                  1647824580  2022-03-21 09:03:00+0800            41  adservice-0,adservice-1,cartservice-0,cartserv...
6                  1647824640  2022-03-21 09:04:00+0800            30  adservice-2,checkoutservice-0,checkoutservice-...
7                  1647824700  2022-03-21 09:05:00+0800            29  adservice-2,checkoutservice-0,checkoutservice-...
8                  1647824940  2022-03-21 09:09:00+0800            29  cartservice-1,cartservice-2,checkoutservice-0,...
9                  1647824820  2022-03-21 09:07:00+0800            27  adservice-0,adservice-1,cartservice-1,checkout...)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse variables from kernel
cas = component_anomaly_summary.copy() if 'component_anomaly_summary' in globals() else pd.DataFrame()
mwa = metric_window_anomalies.copy() if 'metric_window_anomalies' in globals() else pd.DataFrame()

# Ensure numeric types
for c in ['total_anomalies','num_kpis_with_anomaly','earliest_anomaly_timestamp','latest_anomaly_timestamp','max_excess_pct']:
    if c in cas.columns:
        cas[c] = pd.to_numeric(cas[c], errors='coerce')

# Target cluster timestamps
cluster_timestamps = [1647824400, 1647826140]

rows = []
for cluster_ts in cluster_timestamps:
    # select components whose earliest_anomaly_timestamp equals cluster_ts
    subset = cas[cas['earliest_anomaly_timestamp'].notna() & (cas['earliest_anomaly_timestamp'].astype(int) == int(cluster_ts))].copy()
    if subset.empty:
        continue
    # sort by total_anomalies desc and take top 10
    subset = subset.sort_values('total_anomalies', ascending=False).head(10)
    # for each component, compute top5 KPIs from metric_window_anomalies within incident window
    for _, comp in subset.iterrows():
        cmdb = comp['cmdb_id']
        # filter metric_window_anomalies for this component
        comp_kpis = mwa[mwa['cmdb_id'] == cmdb].copy()
        if comp_kpis.empty:
            top5_str = ""
        else:
            # compute anomaly sum per kpi (high+low) and aggregate in case duplicates
            comp_kpis['anomaly_sum'] = comp_kpis[['anomaly_count_high','anomaly_count_low']].fillna(0).sum(axis=1).astype(int)
            agg_kpis = (
                comp_kpis
                .groupby('kpi_name', as_index=False)['anomaly_sum']
                .sum()
                .sort_values('anomaly_sum', ascending=False)
            )
            # format top 5 as "kpi_name(count)"
            top5 = agg_kpis.head(5)
            top5_list = [f"{r['kpi_name']}({int(r['anomaly_sum'])})" for _, r in top5.iterrows()]
            top5_str = ",".join(top5_list)
        rows.append({
            'cluster_ts': int(cluster_ts),
            'cmdb_id': cmdb,
            'total_anomalies': int(comp['total_anomalies']) if not pd.isna(comp['total_anomalies']) else 0,
            'num_kpis_with_anomaly': int(comp['num_kpis_with_anomaly']) if not pd.isna(comp['num_kpis_with_anomaly']) else 0,
            'earliest_anomaly_timestamp': int(comp['earliest_anomaly_timestamp']) if not pd.isna(comp['earliest_anomaly_timestamp']) else pd.NA,
            'latest_anomaly_timestamp': int(comp['latest_anomaly_timestamp']) if not pd.isna(comp['latest_anomaly_timestamp']) else pd.NA,
            'max_excess_pct': float(comp['max_excess_pct']) if not pd.isna(comp['max_excess_pct']) else np.nan,
            'top5_kpis_with_counts': top5_str
        })

# Build DataFrame, store full result in kernel variable
cluster_candidate_components = pd.DataFrame(rows, columns=[
    'cluster_ts','cmdb_id','total_anomalies','num_kpis_with_anomaly',
    'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_excess_pct','top5_kpis_with_counts'
])

# Sort as requested: by total_anomalies desc then max_excess_pct desc, limit to 20 rows
cluster_candidate_components = cluster_candidate_components.sort_values(['total_anomalies','max_excess_pct'], ascending=[False, False]).head(20).reset_index(drop=True)

# Display compact result variable
cluster_candidate_components

```
Out[7]:
```
```python
# Summarize cluster candidate components (plain English) using existing kernel variable
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse cluster_candidate_components from kernel
ccc = cluster_candidate_components.copy() if 'cluster_candidate_components' in globals() else pd.DataFrame()

def ts_to_local(ts):
    try:
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%%m-%d %H:%M:%S%z')  # placeholder, will fix below
    except Exception:
        return "N/A"

# proper converter
def ts_to_local(ts):
    if pd.isna(ts):
        return "N/A"
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

if ccc.empty:
    summary = "No cluster candidate components found in 'cluster_candidate_components'."
else:
    lines = []
    lines.append("Cluster candidate components summary (top candidates per cluster):")
    for cluster_ts, group in ccc.groupby('cluster_ts'):
        local = ts_to_local(cluster_ts)
        lines.append(f"\nCluster {cluster_ts} ({local} UTC+8):")
        # ensure sorted as stored
        for _, r in group.iterrows():
            cmdb = r['cmdb_id']
            total = int(r['total_anomalies']) if not pd.isna(r['total_anomalies']) else 0
            num_kpis = int(r['num_kpis_with_anomaly']) if not pd.isna(r['num_kpis_with_anomaly']) else 0
            earliest = ts_to_local(r['earliest_anomaly_timestamp']) if not pd.isna(r['earliest_anomaly_timestamp']) else "N/A"
            latest = ts_to_local(r['latest_anomaly_timestamp']) if not pd.isna(r['latest_anomaly_timestamp']) else "N/A"
            max_excess = r['max_excess_pct']
            max_excess_s = f"{max_excess:.2f}" if pd.notna(max_excess) else "N/A"
            top5 = r.get('top5_kpis_with_counts', '')
            # truncate top5 if very long
            if isinstance(top5, str) and len(top5) > 200:
                top5 = top5[:197] + "..."
            lines.append(f"- {cmdb}: total_anomalies={total}, kpis_with_anomaly={num_kpis}, earliest={earliest}, latest={latest}, max_excess_pct={max_excess_s}, top5_kpis={top5}")
    lines.append("\nNotes:")
    lines.append("- Results come from components whose earliest anomaly timestamp matched the cluster timestamps (09:00 and 09:29 UTC+8).")
    lines.append("- 'top5_kpis_with_counts' lists the component's top 5 KPIs in the incident window formatted as kpi_name(count).")
    summary = "\n".join(lines)

# Display the concise summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cluster_ts                   cmdb_id  total_anomalies  num_kpis_with_anomaly  earliest_anomaly_timestamp  latest_anomaly_timestamp  max_excess_pct                              top5_kpis_with_counts
0  1647824400                 adservice            18906                    341                  1647824400                1647826200       10.000000  runtime.jvm_threads_state.WAITING(62),runtime....
1  1647824400                adservice2            17707                    333                  1647824400                1647826200        6.220873  runtime.jvm_threads_state.TERMINATED(62),runti...
2  1647824400               frontend2-0            11723                    200                  1647824400                1647826200       16.580427  container.node-6.container_cpu_cfs_throttled_p...
3  1647824400        checkoutservice2-0             8352                    148                  1647824400                1647826200      336.000000  container.node-6.container_cpu_load_average_10...
4  1647824400                frontend-1             8068                    188                  1647824400                1647826200        4.939244  container.node-5.container_cpu_cfs_throttled_p...
5  1647824400                frontend-2             7964                    188                  1647824400                1647826200        4.067466  container.node-5.container_cpu_cfs_throttled_p...
6  1647824400                frontend-0             7927                    188                  1647824400                1647826200        2.064247  container.node-6.container_cpu_cfs_throttled_p...
7  1647824400  recommendationservice2-0             5701                    102                  1647824400                1647826200        1.778972  container.node-6.container_cpu_cfs_throttled_p...
8  1647824400         checkoutservice-1             5522                    144                  1647824400                1647826200       97.333333  container.node-6.container_cpu_load_average_10...
9  1647824400  productcatalogservice2-0             5396                    104                  1647824400                1647826200       26.160000  container.node-6.container_cpu_load_average_10...```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np

# Reuse trace_window_anomalies from kernel
twa = trace_window_anomalies.copy() if 'trace_window_anomalies' in globals() else pd.DataFrame()

# Ensure numeric types for sorting and filtering
for col in ['earliest_anomaly_timestamp','anomaly_count_high','anomaly_count_low','max_value_in_window']:
    if col in twa.columns:
        twa[col] = pd.to_numeric(twa[col], errors='coerce')

# Filter rows where earliest_anomaly_timestamp equals the cluster timestamp 1647824400
cluster_ts = 1647824400
twa_0900 = twa[twa['earliest_anomaly_timestamp'] == cluster_ts].copy()

# Compute total anomalies for sorting
twa_0900['total_anomalies'] = twa_0900[['anomaly_count_high','anomaly_count_low']].fillna(0).sum(axis=1).astype(int)

# Sort by total_anomalies desc, then by max_value_in_window desc
twa_0900_sorted = twa_0900.sort_values(['total_anomalies','max_value_in_window'], ascending=[False, False])

# Select requested columns and limit to top 20
trace_anomalies_0900_top20 = twa_0900_sorted[[
    'cmdb_id','trace_name','total_points_in_window','anomaly_count_high','anomaly_count_low',
    'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','p95','p5'
]].head(20).reset_index(drop=True)

# Display the compact top-20 result (variable name as output)
trace_anomalies_0900_top20

```
Out[8]:
```
```python
import pandas as pd
import pytz

# Reuse the filtered DataFrame from the kernel
df = trace_anomalies_0900_top20.copy() if 'trace_anomalies_0900_top20' in globals() else pd.DataFrame()

tz = pytz.timezone('Asia/Shanghai')

def ts_to_local(ts):
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

if df.empty:
    summary = "No trace anomalies found for earliest_anomaly_timestamp=1647824400."
else:
    n = df.shape[0]
    unique_cmdbs = df['cmdb_id'].unique().tolist()
    sample_cmdbs = ",".join(unique_cmdbs[:10])
    # common trace name patterns
    trace_name_counts = df['trace_name'].value_counts().head(10).to_dict()
    # check p95/p5 and values
    all_p95_zero = (df['p95'].fillna(0) == 0).all()
    all_p5_zero = (df['p5'].fillna(0) == 0).all()
    all_values_zero = (df['max_value_in_window'].fillna(0) == 0).all() and (df['min_value_in_window'].fillna(0) == 0).all()
    earliest_local = ts_to_local(1647824400)
    latest_ts = int(df['latest_anomaly_timestamp'].max()) if 'latest_anomaly_timestamp' in df.columns else None
    latest_local = ts_to_local(latest_ts) if latest_ts else "N/A"

    summary_lines = [
        f"Trace anomalies for cluster 1647824400 ({earliest_local} UTC+8) — top {n} rows returned.",
        f"- Sample cmdb_ids: {sample_cmdbs}",
        f"- Common trace names (top examples): {', '.join([f'{k}({v})' for k,v in trace_name_counts.items()])}",
        f"- Window points per series: typical total_points_in_window = {int(df['total_points_in_window'].mode().iat[0])} (all shown rows have 30).",
        f"- Anomaly counts in these rows: anomaly_count_high = {int(df['anomaly_count_high'].mode().iat[0])}, anomaly_count_low = {int(df['anomaly_count_low'].mode().iat[0])}.",
        f"- Time range in these rows: earliest = {earliest_local}, latest = {latest_local}.",
        f"- Observed max/min values in these rows: max_value_in_window = {df['max_value_in_window'].unique()[:5].tolist()}, p95 values sample = {df['p95'].unique()[:5].tolist()}, p5 values sample = {df['p5'].unique()[:5].tolist()}."
    ]

    if all_p95_zero and all_p5_zero and all_values_zero:
        summary_lines.append(
            "- Important note: p95 and p5 are 0 and observed values are also 0 for these series, "
            "so every point satisfies both value >= p95 and value <= p5. This leads to anomaly_count_high and anomaly_count_low "
            "being equal to the number of points and is likely a degenerate/zero-inflated case. Treat these flags cautiously."
        )
    else:
        summary_lines.append("- p95/p5/value patterns vary; anomalies appear meaningful for some series.")

    summary_lines.append("- Recommendation: inspect raw trace error_rate semantics and consider alternative thresholding (e.g., treat error_rate>0 as anomalous) or examine raw traces for true errors.")
    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                        trace_name  total_points_in_window  anomaly_count_high  anomaly_count_low  earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window  p95   p5
0     adservice-0  trace.from_frontend-0.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
1     adservice-0  trace.from_frontend-1.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
2     adservice-0  trace.from_frontend-2.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
3     adservice-1  trace.from_frontend-0.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
4     adservice-1  trace.from_frontend-1.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
5     adservice-1  trace.from_frontend-2.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
6     adservice-2  trace.from_frontend-0.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
7     adservice-2  trace.from_frontend-1.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
8     adservice-2  trace.from_frontend-2.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
9   cartservice-0  trace.from_frontend-0.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
10  cartservice-0  trace.from_frontend-1.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
11  cartservice-0  trace.from_frontend-2.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
12  cartservice-0             trace.self.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
13  cartservice-1  trace.from_frontend-0.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
14  cartservice-1  trace.from_frontend-1.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
15  cartservice-1  trace.from_frontend-2.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
16  cartservice-1             trace.self.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
17  cartservice-2  trace.from_frontend-0.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
18  cartservice-2  trace.from_frontend-1.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0
19  cartservice-2  trace.from_frontend-2.error_rate                      30                  30                 30                  1647824400                1647826140                  0.0  0.0  0.0```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np

# Reuse existing kernel variables
cas = component_anomaly_summary.copy() if 'component_anomaly_summary' in globals() else pd.DataFrame()
mwa = metric_window_anomalies.copy() if 'metric_window_anomalies' in globals() else pd.DataFrame()

# Ensure numeric types
for col in ['earliest_anomaly_timestamp','latest_anomaly_timestamp','total_anomalies','num_kpis_with_anomaly']:
    if col in cas.columns:
        cas[col] = pd.to_numeric(cas[col], errors='coerce')

for col in ['anomaly_count_high','anomaly_count_low','max_value_in_window','p95','p5']:
    if col in mwa.columns:
        mwa[col] = pd.to_numeric(mwa[col], errors='coerce')

# Target cluster timestamps
cluster_timestamps = [1647824400, 1647826140]

rows = []
for cluster_ts in cluster_timestamps:
    # select components whose earliest_anomaly_timestamp equals cluster_ts
    subset = cas[cas['earliest_anomaly_timestamp'].notna() & (cas['earliest_anomaly_timestamp'].astype(int) == int(cluster_ts))].copy()
    if subset.empty:
        continue
    subset_top = subset.sort_values('total_anomalies', ascending=False).head(10)

    for _, comp in subset_top.iterrows():
        cmdb = comp['cmdb_id']
        # fetch KPI rows for this component from metric_window_anomalies within incident window
        comp_kpis = mwa[mwa['cmdb_id'] == cmdb].copy()
        if comp_kpis.empty:
            top5_str = ""
            top_kpi_max_value = np.nan
            top_kpi_p95 = np.nan
            top_kpi_p5 = np.nan
        else:
            comp_kpis['anomaly_sum'] = comp_kpis[['anomaly_count_high','anomaly_count_low']].fillna(0).sum(axis=1).astype(int)
            agg_kpis = (
                comp_kpis
                .groupby('kpi_name', as_index=False)
                .agg(anomaly_sum=('anomaly_sum','sum'),
                     max_value_in_window=('max_value_in_window','max'),
                     p95=('p95','first'),
                     p5=('p5','first'))
                .sort_values('anomaly_sum', ascending=False)
            )
            top5 = agg_kpis.head(5)
            top5_list = [f"{r['kpi_name']}({int(r['anomaly_sum'])})" for _, r in top5.iterrows()]
            top5_str = ",".join(top5_list)
            # use the top KPI's metrics for the component-level max_value/p95/p5 fields
            if not top5.empty:
                top_kpi_max_value = top5.iloc[0]['max_value_in_window']
                top_kpi_p95 = top5.iloc[0]['p95']
                top_kpi_p5 = top5.iloc[0]['p5']
            else:
                top_kpi_max_value = np.nan
                top_kpi_p95 = np.nan
                top_kpi_p5 = np.nan

        rows.append({
            'cluster_ts': int(cluster_ts),
            'cmdb_id': cmdb,
            'total_anomalies': int(comp['total_anomalies']) if not pd.isna(comp['total_anomalies']) else 0,
            'num_kpis_with_anomaly': int(comp['num_kpis_with_anomaly']) if not pd.isna(comp['num_kpis_with_anomaly']) else 0,
            'earliest_anomaly_timestamp': int(comp['earliest_anomaly_timestamp']) if not pd.isna(comp['earliest_anomaly_timestamp']) else pd.NA,
            'latest_anomaly_timestamp': int(comp['latest_anomaly_timestamp']) if not pd.isna(comp['latest_anomaly_timestamp']) else pd.NA,
            'max_value_in_window': top_kpi_max_value,
            'p95': top_kpi_p95,
            'p5': top_kpi_p5,
            'top5_kpis_with_counts': top5_str
        })

# Build DataFrame and store full result
cluster_metric_details = pd.DataFrame(rows, columns=[
    'cluster_ts','cmdb_id','total_anomalies','num_kpis_with_anomaly',
    'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','p95','p5','top5_kpis_with_counts'
])

# Sort by total_anomalies desc then max_value_in_window desc and limit to 20 rows (up to 10 per cluster as constructed)
cluster_metric_details = cluster_metric_details.sort_values(['total_anomalies','max_value_in_window'], ascending=[False, False]).reset_index(drop=True)
cluster_metric_details_top20 = cluster_metric_details.head(20)

# Return the compact top-20 table (and keep full 'cluster_metric_details' in kernel)
cluster_metric_details_top20

```
Out[9]:
```
```python
import pandas as pd
import pytz

# Reuse cluster_metric_details (full) or the top-20 view if available
df = None
if 'cluster_metric_details' in globals() and isinstance(cluster_metric_details, pd.DataFrame):
    df = cluster_metric_details.copy()
elif 'cluster_metric_details_top20' in globals() and isinstance(cluster_metric_details_top20, pd.DataFrame):
    df = cluster_metric_details_top20.copy()
else:
    df = pd.DataFrame()

tz = pytz.timezone('Asia/Shanghai')

def to_local(ts):
    try:
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    except Exception:
        return "N/A"

if df.empty:
    summary = "No cluster metric details available."
else:
    # Limit to at most 20 rows (up to 10 per cluster)
    df = df.head(20)
    lines = []
    lines.append("Cluster metric candidates (top components per cluster):")
    for _, r in df.iterrows():
        cluster_ts = int(r['cluster_ts'])
        cluster_local = to_local(cluster_ts)
        cmdb = r['cmdb_id']
        total = int(r['total_anomalies']) if pd.notna(r['total_anomalies']) else 0
        num_kpis = int(r['num_kpis_with_anomaly']) if pd.notna(r['num_kpis_with_anomaly']) else 0
        earliest = to_local(r['earliest_anomaly_timestamp']) if pd.notna(r.get('earliest_anomaly_timestamp', None)) else "N/A"
        latest = to_local(r['latest_anomaly_timestamp']) if pd.notna(r.get('latest_anomaly_timestamp', None)) else "N/A"
        maxv = r.get('max_value_in_window', None)
        p95 = r.get('p95', None)
        p5 = r.get('p5', None)
        kpistr = r.get('top5_kpis_with_counts', '')
        if isinstance(kpistr, str) and len(kpistr) > 160:
            kpistr = kpistr[:157] + "..."
        lines.append(
            f"- Cluster {cluster_ts} ({cluster_local}): {cmdb} | total_anomalies={total}, kpis_with_anomaly={num_kpis}, "
            f"earliest={earliest}, latest={latest}, max_value={maxv}, p95={p95}, p5={p5}, top5_kpis=[{kpistr}]"
        )
    lines.append("\nNote: Times are shown in UTC+8. Full table stored in 'cluster_metric_details'.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cluster_ts                   cmdb_id  total_anomalies  num_kpis_with_anomaly  earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window  p95   p5                              top5_kpis_with_counts
0  1647824400                 adservice            18906                    341                  1647824400                1647826200                  7.0  7.0  7.0  runtime.jvm_threads_state.WAITING(62),runtime....
1  1647824400                adservice2            17707                    333                  1647824400                1647826200                  0.0  0.0  0.0  runtime.jvm_threads_state.TERMINATED(62),runti...
2  1647824400               frontend2-0            11723                    200                  1647824400                1647826200                  0.0  0.0  0.0  container.node-6.container_cpu_cfs_throttled_p...
3  1647824400        checkoutservice2-0             8352                    148                  1647824400                1647826200                  0.0  0.0  0.0  container.node-6.container_cpu_load_average_10...
4  1647824400                frontend-1             8068                    188                  1647824400                1647826200                  0.0  0.0  0.0  container.node-5.container_cpu_cfs_throttled_p...
5  1647824400                frontend-2             7964                    188                  1647824400                1647826200                  0.0  0.0  0.0  container.node-5.container_cpu_cfs_throttled_p...
6  1647824400                frontend-0             7927                    188                  1647824400                1647826200                  0.0  0.0  0.0  container.node-6.container_cpu_cfs_throttled_p...
7  1647824400  recommendationservice2-0             5701                    102                  1647824400                1647826200                  0.0  0.0  0.0  container.node-6.container_cpu_cfs_throttled_p...
8  1647824400         checkoutservice-1             5522                    144                  1647824400                1647826200                  0.0  0.0  0.0  container.node-6.container_cpu_load_average_10...
9  1647824400  productcatalogservice2-0             5396                    104                  1647824400                1647826200                  0.0  0.0  0.0  container.node-6.container_cpu_load_average_10...```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np

# Reuse metric_window_anomalies from kernel
mwa = metric_window_anomalies.copy() if 'metric_window_anomalies' in globals() else pd.DataFrame()

# Ensure numeric types
for col in ['anomaly_count_high','anomaly_count_low','earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','p95','p5']:
    if col in mwa.columns:
        mwa[col] = pd.to_numeric(mwa[col], errors='coerce')

# Target cluster timestamp (09:29 UTC+8)
cluster_ts = 1647826140

# Filter rows whose earliest_anomaly_timestamp equals the cluster timestamp
mwa_0929 = mwa[mwa['earliest_anomaly_timestamp'] == cluster_ts].copy()

# If empty, create empty result
if mwa_0929.empty:
    cluster_0929_metric_candidates = pd.DataFrame(columns=[
        'cmdb_id','total_anomalies','num_kpis_with_anomaly',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp',
        'max_value_in_window','p95','p5','top5_kpis_with_counts'
    ])
else:
    # compute anomaly_sum per row
    mwa_0929['anomaly_sum'] = mwa_0929[['anomaly_count_high','anomaly_count_low']].fillna(0).sum(axis=1).astype(int)

    def aggregate_group(g):
        total_anomalies = int(g['anomaly_sum'].sum())
        num_kpis_with_anomaly = int(g.loc[g['anomaly_sum']>0, 'kpi_name'].nunique())
        earliest = int(g['earliest_anomaly_timestamp'].dropna().min()) if g['earliest_anomaly_timestamp'].dropna().size>0 else pd.NA
        latest = int(g['latest_anomaly_timestamp'].dropna().max()) if g['latest_anomaly_timestamp'].dropna().size>0 else pd.NA
        max_value = g['max_value_in_window'].dropna().max() if g['max_value_in_window'].dropna().size>0 else np.nan
        p95_max = g['p95'].dropna().max() if g['p95'].dropna().size>0 else np.nan
        p5_min = g['p5'].dropna().min() if g['p5'].dropna().size>0 else np.nan
        # top 5 kpis by anomaly_sum for this cmdb_id
        topk = (
            g.groupby('kpi_name', as_index=False)['anomaly_sum']
             .sum()
             .sort_values('anomaly_sum', ascending=False)
             .head(5)
        )
        top5_list = [f"{row['kpi_name']}({int(row['anomaly_sum'])})" for _, row in topk.iterrows()]
        top5_str = ",".join(top5_list)
        return pd.Series({
            'total_anomalies': total_anomalies,
            'num_kpis_with_anomaly': num_kpis_with_anomaly,
            'earliest_anomaly_timestamp': earliest,
            'latest_anomaly_timestamp': latest,
            'max_value_in_window': max_value,
            'p95': p95_max,
            'p5': p5_min,
            'top5_kpis_with_counts': top5_str
        })

    agg = mwa_0929.groupby('cmdb_id').apply(aggregate_group).reset_index()
    cluster_0929_metric_candidates = agg.copy()

# Sort by total_anomalies desc and limit to top 20 for compact display
cluster_0929_metric_candidates_sorted = cluster_0929_metric_candidates.sort_values('total_anomalies', ascending=False).head(20).reset_index(drop=True)

# Keep full aggregated result in kernel variable and display compact top-20
cluster_0929_metric_candidates, cluster_0929_metric_candidates_sorted

```
Out[10]:
```
```python
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse aggregated result from kernel
df = cluster_0929_metric_candidates.copy() if 'cluster_0929_metric_candidates' in globals() else pd.DataFrame()

def to_local(ts):
    if pd.isna(ts):
        return "N/A"
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

if df.empty:
    summary = "No metric candidates found for cluster 1647826140 (2022-03-21 09:29:00+08)."
else:
    # Provide concise summary of top rows (dataset is small here)
    lines = []
    lines.append("Cluster 1647826140 (2022-03-21 09:29:00+08) metric candidates:")
    for _, r in df.iterrows():
        cmdb = r['cmdb_id']
        total = int(r['total_anomalies'])
        num_kpis = int(r['num_kpis_with_anomaly'])
        earliest = to_local(r['earliest_anomaly_timestamp'])
        latest = to_local(r['latest_anomaly_timestamp'])
        maxv = float(r['max_value_in_window']) if pd.notna(r['max_value_in_window']) else None
        p95 = float(r['p95']) if pd.notna(r['p95']) else None
        p5 = float(r['p5']) if pd.notna(r['p5']) else None
        top5 = r.get('top5_kpis_with_counts', '')
        lines.append(
            f"- {cmdb}: total_anomalies={total}, num_kpis_with_anomaly={num_kpis}, "
            f"earliest={earliest}, latest={latest}, max_value={maxv}, p95={p95}, p5={p5}, top5_kpis=[{top5}]"
        )
    lines.append("\nFull aggregated table stored in 'cluster_0929_metric_candidates'.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  total_anomalies  num_kpis_with_anomaly  earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window         p95         p5                              top5_kpis_with_counts
0  adservice2                4                      2                  1647826140                1647826200         4.665471e+07  42603928.0  40.018263  runtime.java_lang_Memory_NonHeapMemoryUsage_us...,       cmdb_id  total_anomalies  num_kpis_with_anomaly  earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window         p95         p5                              top5_kpis_with_counts
0  adservice2                4                      2                  1647826140                1647826200         4.665471e+07  42603928.0  40.018263  runtime.java_lang_Memory_NonHeapMemoryUsage_us...)```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz
import numpy as np

# Reuse log_df if present, otherwise load
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

# Ensure correct dtypes
log_df['timestamp'] = log_df['timestamp'].astype('int64')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Timezone and incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = pd.Timestamp('2022-03-21 09:00:00').tz_localize(tz)
end_local = pd.Timestamp('2022-03-21 09:30:00').tz_localize(tz)
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts = int(end_local.astimezone(pytz.utc).timestamp())

# 1) Compute global thresholds (P95, P5, count) across full log data BEFORE filtering
log_thresholds = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(
        count=lambda x: x.dropna().shape[0],
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05)
    )
)

# 2) Filter log data to the incident window (inclusive)
log_window = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts)].copy()

# 3) Join filtered rows with global thresholds on (cmdb_id, log_name)
merged = log_window.merge(log_thresholds, on=['cmdb_id', 'log_name'], how='inner')

# 4) Compute anomaly flags and aggregate per (cmdb_id, log_name)
if merged.empty:
    log_window_anomalies = pd.DataFrame(columns=[
        'cmdb_id','log_name','total_points_in_window','anomaly_count_high','anomaly_count_low',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','min_value_in_window','p95','p5'
    ])
else:
    merged['is_high'] = merged['value'] >= merged['p95']
    merged['is_low'] = merged['value'] <= merged['p5']
    merged['is_anomaly'] = merged['is_high'] | merged['is_low']
    merged['anomaly_ts'] = merged['timestamp'].where(merged['is_anomaly'], pd.NA)

    grouped = (
        merged
        .groupby(['cmdb_id','log_name','p95','p5'], as_index=False)
        .agg(
            total_points_in_window=('value','count'),
            anomaly_count_high=('is_high','sum'),
            anomaly_count_low=('is_low','sum'),
            earliest_anomaly_timestamp=('anomaly_ts','min'),
            latest_anomaly_timestamp=('anomaly_ts','max'),
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min')
        )
    )

    # Normalize types
    def to_int_or_na(x):
        if pd.isna(x):
            return pd.NA
        return int(x)

    grouped['earliest_anomaly_timestamp'] = grouped['earliest_anomaly_timestamp'].apply(to_int_or_na)
    grouped['latest_anomaly_timestamp'] = grouped['latest_anomaly_timestamp'].apply(to_int_or_na)
    grouped['total_points_in_window'] = grouped['total_points_in_window'].astype(int)
    grouped['anomaly_count_high'] = grouped['anomaly_count_high'].astype(int)
    grouped['anomaly_count_low'] = grouped['anomaly_count_low'].astype(int)

    log_window_anomalies = grouped[[
        'cmdb_id','log_name','total_points_in_window','anomaly_count_high','anomaly_count_low',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','min_value_in_window','p95','p5'
    ]].copy()

# 5) Prepare compact top-20 table sorted by total anomalies desc then max_value_in_window desc
log_window_anomalies['total_anomalies'] = log_window_anomalies['anomaly_count_high'].fillna(0).astype(int) + log_window_anomalies['anomaly_count_low'].fillna(0).astype(int)
log_window_anomalies_top20 = log_window_anomalies.sort_values(['total_anomalies','max_value_in_window'], ascending=[False, False]).head(20)

# Keep variables in kernel: log_thresholds, log_window_anomalies
log_thresholds, log_window_anomalies_top20

```
Out[11]:
```
```python
# Concise plain-English summary based on the computed log thresholds and window anomalies.
import pandas as pd

# Reuse kernel variables
lt = log_thresholds if 'log_thresholds' in globals() else None
lwa = log_window_anomalies if 'log_window_anomalies' in globals() else None
lwa_top20 = log_window_anomalies_top20 if 'log_window_anomalies_top20' in globals() else None

if lt is None:
    summary = "log_thresholds not found in kernel."
elif lwa is None:
    summary = "log_window_anomalies not found in kernel."
else:
    total_threshold_series = int(lt.shape[0])
    total_window_series = int(lwa.shape[0])
    # Check prevalence of zero thresholds in top anomalies
    top_sample = lwa_top20.head(10) if lwa_top20 is not None else lwa.head(10)
    zero_p95_p5 = ((top_sample['p95'].fillna(0) == 0) & (top_sample['p5'].fillna(0) == 0)).sum() if not top_sample.empty else 0

    summary_lines = [
        f"Log thresholds and window anomalies (window: 2022-03-21 09:00–09:30 UTC+8):",
        f"- Global thresholds computed for {total_threshold_series} (cmdb_id, log_name) series (stored in 'log_thresholds').",
        f"- Aggregated anomalies in-window stored in 'log_window_anomalies' for {total_window_series} series.",
        f"- Top anomalies (shown in 'log_window_anomalies_top20') include many 'log.error_count' series where p95=0 and p5=0; in those cases every point in the window is flagged as both >=p95 and <=p5.",
        f"  Example pattern: several services (e.g., adservice-1, cartservice-*, frontend-*) have log.error_count with anomaly_count_high and anomaly_count_low equal to the number of points in-window.",
        f"- Non-zero-threshold examples exist for 'log.row_count' (p95/p5 > 0) — these behave more like typical numeric thresholds.",
        "",
        "Implication: a large number of log.error_count series are zero-inflated (p95==p5==0); treat those anomaly flags cautiously (they may be degenerate).",
        "Recommendation: inspect raw log semantics for error_count (consider using >0 as a rule for error presence) and prioritize non-degenerate series (e.g., log.row_count) for metric-based investigation."
    ]
    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id         log_name  count   p95    p5
0         adservice-0  log.error_count     30   0.0   0.0
1         adservice-0    log.row_count     30  76.0  48.5
2         adservice-1  log.error_count     31   0.0   0.0
3         adservice-1    log.row_count     31  63.0  31.0
4         adservice-2  log.error_count     30   0.0   0.0
..                ...              ...    ...   ...   ...
57  shippingservice-0    log.row_count     31  60.5  26.0
58  shippingservice-1  log.error_count     30   0.0   0.0
59  shippingservice-1    log.row_count     30  58.0  30.0
60  shippingservice-2  log.error_count     30   0.0   0.0
61  shippingservice-2    log.row_count     30  59.2  31.7

[62 rows x 5 columns],                     cmdb_id         log_name  total_points_in_window  anomaly_count_high  anomaly_count_low  earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window  min_value_in_window  p95   p5  total_anomalies
2               adservice-1  log.error_count                      31                  31                 31                  1647824400                1647826200                  0.0                  0.0  0.0  0.0               62
6             cartservice-0  log.error_count                      31                  31                 31                  1647824400                1647826200                  0.0                  0.0  0.0  0.0               62
8             cartservice-1  log.error_count                      31                  31                 31                  1647824400                1647826200                  0.0                  0.0  0.0  0.0               62
10            cartservice-2  log.error_count                      31                  31                 31                  1647824400                1647826200                  0.0                  0.0  0.0  0.0               62
18        currencyservice-0  log.error_count                      31                  31                 31                  1647824400                1647826200                  0.0                  0.0  0.0  0.0               62
20        currencyservice-1  log.error_count                      31                  31                 31                  1647824400                1647826200                  0.0                  0.0  0.0  0.0               62
22        currencyservice-2  log.error_count                      31                  31                 31                  1647824400                1647826200                  0.0                  0.0  0.0  0.0               62
30               frontend-0  log.error_count                      31                  31                 31                  1647824400                1647826200                  0.0                  0.0  0.0  0.0               62
32               frontend-1  log.error_count                      31                  31                 31                  1647824400                1647826200                  0.0                  0.0  0.0  0.0               62
34               frontend-2  log.error_count                      31                  31                 31                  1647824400                1647826200                  0.0                  0.0  0.0  0.0               62
46  productcatalogservice-2  log.error_count                      31                  31                 31                  1647824400                1647826200                  0.0                  0.0  0.0  0.0               62
48  recommendationservice-0  log.error_count                      31                  31                 31                  1647824400                1647826200                  0.0                  0.0  0.0  0.0               62
50  recommendationservice-1  log.error_count                      31                  31                 31                  1647824400                1647826200                  0.0                  0.0  0.0  0.0               62
52  recommendationservice-2  log.error_count                      31                  31                 31                  1647824400                1647826200                  0.0                  0.0  0.0  0.0               62
56        shippingservice-0  log.error_count                      31                  31                 31                  1647824400                1647826200                  0.0                  0.0  0.0  0.0               62
0               adservice-0  log.error_count                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
4               adservice-2  log.error_count                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
12        checkoutservice-0  log.error_count                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
24           emailservice-0  log.error_count                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60
26           emailservice-1  log.error_count                      30                  30                 30                  1647824400                1647826140                  0.0                  0.0  0.0  0.0               60)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[12]:

import pandas as pd
import numpy as np

# Reuse metric_window_anomalies from kernel
mwa = metric_window_anomalies.copy() if 'metric_window_anomalies' in globals() else pd.DataFrame()

# Ensure numeric columns
for c in ['anomaly_count_high','anomaly_count_low','earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','p95','p5']:
    if c in mwa.columns:
        mwa[c] = pd.to_numeric(mwa[c], errors='coerce')

# Part 1: Node-level check for 09:00 cluster (1647824400)
cluster_ts_0900 = 1647824400
nodes_filter = mwa[
    (mwa['earliest_anomaly_timestamp'] == cluster_ts_0900) &
    (mwa['cmdb_id'].str.startswith('node-', na=False))
].copy()

if nodes_filter.empty:
    nodes_0900_candidates = pd.DataFrame(columns=[
        'cmdb_id','total_anomalies','num_kpis_with_anomaly',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp',
        'max_excess_pct','top5_kpis_with_counts'
    ])
else:
    # compute per-row anomaly sum
    nodes_filter['anomaly_sum'] = nodes_filter[['anomaly_count_high','anomaly_count_low']].fillna(0).sum(axis=1).astype(int)

    # compute per-row excess_pct where p95 != 0
    def calc_excess(row):
        p95 = row['p95']
        maxv = row['max_value_in_window']
        if pd.isna(p95) or p95 == 0 or pd.isna(maxv):
            return np.nan
        return (maxv - p95) / p95

    nodes_filter['excess_pct'] = nodes_filter.apply(calc_excess, axis=1)

    # aggregate by node
    def agg_node(g):
        total_anomalies = int(g['anomaly_sum'].sum())
        num_kpis_with_anomaly = int(g.loc[g['anomaly_sum']>0, 'kpi_name'].nunique())
        earliest = int(g['earliest_anomaly_timestamp'].dropna().min()) if g['earliest_anomaly_timestamp'].dropna().size>0 else pd.NA
        latest = int(g['latest_anomaly_timestamp'].dropna().max()) if g['latest_anomaly_timestamp'].dropna().size>0 else pd.NA
        max_excess = float(g['excess_pct'].dropna().max()) if g['excess_pct'].dropna().size>0 else np.nan
        # top5 kpis by anomaly_sum
        topk = (
            g.groupby('kpi_name', as_index=False)['anomaly_sum']
             .sum()
             .sort_values('anomaly_sum', ascending=False)
             .head(5)
        )
        top5_list = [f"{row['kpi_name']}({int(row['anomaly_sum'])})" for _, row in topk.iterrows()]
        top5_str = ",".join(top5_list)
        return pd.Series({
            'total_anomalies': total_anomalies,
            'num_kpis_with_anomaly': num_kpis_with_anomaly,
            'earliest_anomaly_timestamp': earliest,
            'latest_anomaly_timestamp': latest,
            'max_excess_pct': max_excess,
            'top5_kpis_with_counts': top5_str
        })

    nodes_0900_candidates = nodes_filter.groupby('cmdb_id').apply(agg_node).reset_index()

# Sort and prepare compact top-10 view
nodes_0900_candidates_sorted = nodes_0900_candidates.sort_values(['total_anomalies','max_excess_pct'], ascending=[False, False]).head(10)

# Part 2: adservice2 deep-check for 09:29 cluster (1647826140)
cluster_ts_0929 = 1647826140
adservice2_0929_kpis = mwa[
    (mwa['cmdb_id'] == 'adservice2') &
    (mwa['earliest_anomaly_timestamp'] == cluster_ts_0929)
].copy()

# Select requested columns and limit to 20 rows
if not adservice2_0929_kpis.empty:
    adservice2_0929_kpis = adservice2_0929_kpis[[
        'cmdb_id','kpi_name','total_points_in_window','anomaly_count_high','anomaly_count_low',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','min_value_in_window','p95','p5'
    ]].head(20).reset_index(drop=True)
else:
    # empty frame with expected columns
    adservice2_0929_kpis = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points_in_window','anomaly_count_high','anomaly_count_low',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','min_value_in_window','p95','p5'
    ])

# Store full aggregated results in kernel variables (already assigned)
# nodes_0900_candidates (full), adservice2_0929_kpis (filtered)
nodes_0900_candidates_sorted, adservice2_0929_kpis

```
Out[12]:
```
```python
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables
nodes_df = nodes_0900_candidates_sorted.copy() if 'nodes_0900_candidates_sorted' in globals() else pd.DataFrame()
ad_kpis = adservice2_0929_kpis.copy() if 'adservice2_0929_kpis' in globals() else pd.DataFrame()

def fmt_ts(ts):
    try:
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    except Exception:
        return "N/A"

lines = []
lines.append("Brief findings from metric-window anomaly checks:\n")

# Part 1: Node-level check at 09:00 cluster
if nodes_df.empty:
    lines.append("1) Nodes (09:00 cluster): no node-level candidate metrics found.")
else:
    lines.append(f"1) Nodes (09:00 cluster = 2022-03-21 09:00:00+08) — top {min(10, len(nodes_df))} nodes by total_anomalies:")
    # show up to 10 nodes
    for _, r in nodes_df.head(10).iterrows():
        cmdb = r['cmdb_id']
        total = int(r['total_anomalies']) if pd.notna(r['total_anomalies']) else 0
        num_kpis = int(r['num_kpis_with_anomaly']) if pd.notna(r['num_kpis_with_anomaly']) else 0
        earliest = fmt_ts(r['earliest_anomaly_timestamp'])
        latest = fmt_ts(r['latest_anomaly_timestamp'])
        max_excess = r['max_excess_pct']
        max_excess_s = f"{max_excess:.3f}" if pd.notna(max_excess) else "N/A"
        top5 = r.get('top5_kpis_with_counts', '')
        if isinstance(top5, str) and len(top5) > 160:
            top5 = top5[:157] + "..."
        lines.append(f"   - {cmdb}: total_anomalies={total}, kpis_with_anomaly={num_kpis}, earliest={earliest}, latest={latest}, max_excess_pct={max_excess_s}, top5_kpis=[{top5}]")

    lines.append("   Observations: node-4/5/6 show the highest anomaly totals; node-6 has very large max_excess_pct (large relative deviation for some KPI(s)), node-5/node-4 also high. Investigate node-level disk/IO and ping/connect KPIs listed in top5.")

# Part 2: adservice2 deep-check at 09:29 cluster
lines.append("\n2) adservice2 (09:29 cluster = 2022-03-21 09:29:00+08): KPI details in-window")
if ad_kpis.empty:
    lines.append("   No adservice2 KPIs found for earliest_anomaly_timestamp = 1647826140.")
else:
    lines.append(f"   Found {len(ad_kpis)} KPI rows (showing up to 20):")
    # show up to 10 concise KPI lines
    for _, r in ad_kpis.head(20).iterrows():
        kpi = r['kpi_name']
        pts = int(r['total_points_in_window']) if pd.notna(r['total_points_in_window']) else 0
        high = int(r['anomaly_count_high']) if pd.notna(r['anomaly_count_high']) else 0
        low = int(r['anomaly_count_low']) if pd.notna(r['anomaly_count_low']) else 0
        earliest = fmt_ts(r['earliest_anomaly_timestamp'])
        latest = fmt_ts(r['latest_anomaly_timestamp'])
        maxv = float(r['max_value_in_window']) if pd.notna(r['max_value_in_window']) else None
        p95 = float(r['p95']) if pd.notna(r['p95']) else None
        p5 = float(r['p5']) if pd.notna(r['p5']) else None
        lines.append(f"   - {kpi}: points={pts}, high={high}, low={low}, earliest={earliest}, latest={latest}, max={maxv}, p95={p95}, p5={p5}")

    lines.append("   Note: these KPIs (e.g., Memory NonHeap used) show a few anomalies at 09:29; investigate those specific memory metrics on adservice2.")

lines.append("\nStored variables for follow-up:")
lines.append("- nodes_0900_candidates (full aggregated node candidates).")
lines.append("- adservice2_0929_kpis (detailed KPI rows for adservice2 at 09:29).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id  total_anomalies  num_kpis_with_anomaly  earliest_anomaly_timestamp  latest_anomaly_timestamp  max_excess_pct                              top5_kpis_with_counts
3  node-4             1530                     31                  1647824400                1647826200        1.500000  ping.can_connect(62),system.io.r_await(62),sys...
4  node-5             1508                     31                  1647824400                1647826200        8.000000  system.disk.readonly(62),system.fs.inodes.in_u...
5  node-6             1506                     45                  1647824400                1647826200     2098.000000  ping.can_connect(62),system.disk.readonly(62),...
1  node-2             1470                     28                  1647824400                1647826200        0.707317  ping.can_connect(62),system.disk.readonly(62),...
2  node-3             1430                     28                  1647824400                1647826200        1.250000  system.disk.readonly(62),system.disk.total(62)...
0  node-1             1278                     26                  1647824400                1647826200        1.820896  ping.can_connect(62),system.disk.readonly(62),...,       cmdb_id                                          kpi_name  total_points_in_window  anomaly_count_high  anomaly_count_low  earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window  min_value_in_window           p95            p5
0  adservice2  runtime.java_lang_Memory_NonHeapMemoryUsage_used                    31.0                 2.0                0.0                1.647826e+09              1.647826e+09         4.665471e+07         4.251629e+07  4.260393e+07  4.196012e+07
1  adservice2                runtime.jvm_memory_MB_used.nonheap                    31.0                 2.0                0.0                1.647826e+09              1.647826e+09         4.449346e+01         4.054703e+01  4.063027e+01  4.001826e+01)```
```